In [1]:
import json
import uproot
from pathlib import Path
from typing import Dict, Set, Tuple

import os
import time
import datetime
import multiprocessing
import numpy as np
import pandas as pd
import uproot
import awkward as ak
from concurrent.futures import ProcessPoolExecutor # Swapped for CPU-bound tasks

### Determine Data Luminosity

In [2]:
# --- Configuration & Constants ---
DATA_BASE_PATH = Path("/home/aegis/ether/Research_HEP/Dataset_ver4/Data/reduce_root")
METADATA_FILE = Path("data_metadata.json")

LUMINOSITY_DICT = {
    "A": 0.5608, "B": 2.0034, "C": 2.9165, "D": 4.7087,
    "E": 1.5031, "F": 3.4567, "G": 3.8969, "I": 5.8896,
    "K": 2.2366, "L": 6.2230
}

PERIODS = list(LUMINOSITY_DICT.keys())

# Using a set literal is slightly faster and cleaner than set([...])
GRL_SET = {
    297730, 298595, 298609, 298633, 298687, 298690, 298771, 298773, 298862, 
    298967, 299055, 299144, 299147, 299184, 299243, 299584, 300279, 300345, 
    300415, 300418, 300487, 300540, 300571, 300600, 300655, 300687, 300784, 
    300800, 300863, 300908, 301912, 301915, 301918, 301932, 301973, 302053, 
    302137, 302265, 302269, 302300, 302347, 302380, 302391, 302393, 302737, 
    302831, 302872, 302919, 302925, 302956, 303007, 303079, 303201, 303208, 
    303264, 303266, 303291, 303304, 303338, 303421, 303499, 303560, 303638, 
    303832, 303846, 303892, 303943, 304006, 304008, 304128, 304178, 304198, 
    304211, 304243, 304308, 304337, 304409, 304431, 304494, 305380, 305543, 
    305571, 305618, 305671, 305674, 305723, 305727, 305735, 305777, 305811, 
    305920, 306269, 306278, 306310, 306384, 306419, 306442, 306448, 306451, 
    307126, 307195, 307259, 307306, 307354, 307358, 307394, 307454, 307514, 
    307539, 307569, 307601, 307619, 307656, 307710, 307716, 307732, 307861, 
    307935, 308047, 308084, 309375, 309390, 309440, 309516, 309640, 309674, 
    309759, 310015, 310247, 310249, 310341, 310370, 310405, 310468, 310473, 
    310634, 310691, 310738, 310809, 310863, 310872, 310969, 311071, 311170, 
    311244, 311287, 311321, 311365, 311402, 311473, 311481
}

# --- Core Functions ---

def get_processed_runs(base_path: Path) -> Dict[str, Set[int]]:
    """Scans the directory structure to find which runs have been processed per period."""
    processed = {}
    if not base_path.exists():
        print(f"Warning: Base path {base_path} does not exist.")
        return processed

    for period_dir in base_path.iterdir():
        if period_dir.is_dir():
            period = period_dir.name
            processed[period] = {
                int(run_dir.name.replace("run", ""))
                for run_dir in period_dir.iterdir() if run_dir.is_dir()
            }
    return processed

def load_metadata(json_path: Path, periods: list) -> Tuple[Dict[str, Set[int]], Dict[int, int]]:
    """
    Parses metadata JSON in a single pass.
    Returns:
        - A dictionary mapping periods to sets of run numbers.
        - A dictionary mapping run numbers to expected event counts.
    """
    with open(json_path, "r") as f:
        metadata = json.load(f)

    runs_by_period = {p: set() for p in periods}
    events_by_run = {}

    for item in metadata:
        run_num = int(item["run_number"])
        events_by_run[run_num] = item["events"]
        
        for p in periods:
            if item["period"].startswith(p):
                runs_by_period[p].add(run_num)

    return runs_by_period, events_by_run

def count_actual_events(run_path: Path) -> int:
    """Safely opens ROOT files in a run directory and sums the entries."""
    num_events = 0
    for file_path in run_path.iterdir():
        if file_path.is_file(): # Can add `and file_path.suffix == '.root'` if needed
            try:
                # Combining path and tree name in uproot.open is often faster
                num_events += uproot.open(f"{file_path}:CollectionTree").num_entries
            except Exception as e:
                # Catching a bare Exception is okay here, but we log the specific error
                print(f"Error reading {file_path.name}: {e}")
    return num_events

def analyze_period(period: str, processed_runs: Set[int], runs_in_period: Set[int], events_by_run: Dict[int, int]):
    """Calculates actual vs expected events and effective luminosity for a specific period."""
    if period not in LUMINOSITY_DICT:
        print(f"Period {period} not found in Luminosity Dictionary.")
        return

    print(f"\n--- Analyzing Period {period} ---")
    print(f"Total possible luminosity: {LUMINOSITY_DICT[period]}")
    
    available_grl = runs_in_period.intersection(GRL_SET)
    print(f"Runs in period {period} that are in the GRL: {len(available_grl)}")

    runs_processed = processed_runs.intersection(available_grl)
    print(f"Runs in period {period} (GRL + Processed): {len(runs_processed)}")

    print("\nAll Available GRL Runs:")
    print(sorted(available_grl))
    print("="*50)

    total_actual_events = 0
    period_path = DATA_BASE_PATH / period

    # Calculate Actual Events
    for run in sorted(runs_processed):
        run_path = period_path / f"run{run}"
        actual_events = count_actual_events(run_path)
        print(f"{run} \t Actual Events: {actual_events}")
        total_actual_events += actual_events

    print(f"Total actual events for period {period}: {total_actual_events}")

    # Calculate Expected Events
    total_expected_events = 0
    for run in available_grl:
        expected_events = events_by_run.get(run, 0)
        total_expected_events += expected_events
        print(f"{run} \t Expected Events: {expected_events}")
    print(f"Total expected events for period {period}: {total_expected_events}")
    print("="*50)

    # Calculate Final Luminosity
    if total_expected_events > 0:
        eff_lumi = (total_actual_events / total_expected_events) * LUMINOSITY_DICT[period]
        print(f"Effective Luminosity processed: {eff_lumi:.4f}")
        return eff_lumi
    else:
        print("Effective Luminosity processed: 0.0000 (No expected events found)")
        return 0

In [3]:
processed_runs_dict = get_processed_runs(DATA_BASE_PATH)
runs_by_period, expected_events_by_run = load_metadata(METADATA_FILE, PERIODS)

In [4]:
total_lumi = 0
for p in PERIODS:
    lumi = analyze_period(
        period=p, 
        processed_runs=processed_runs_dict.get(p, set()), 
        runs_in_period=runs_by_period.get(p, set()), 
        events_by_run=expected_events_by_run
    )
    total_lumi = total_lumi + lumi


--- Analyzing Period A ---
Total possible luminosity: 0.5608
Runs in period A that are in the GRL: 17
Runs in period A (GRL + Processed): 17

All Available GRL Runs:
[297730, 298595, 298609, 298633, 298687, 298690, 298771, 298773, 298862, 298967, 299055, 299144, 299147, 299184, 299243, 299584, 300279]
297730 	 Actual Events: 14666349
298595 	 Actual Events: 3133859
298609 	 Actual Events: 6288037
298633 	 Actual Events: 13229931
298687 	 Actual Events: 17971981
298690 	 Actual Events: 1258445
298771 	 Actual Events: 7061356
298773 	 Actual Events: 6837614
298862 	 Actual Events: 20220154
298967 	 Actual Events: 27997456
Error reading root_299055_26.root: not found: 'CollectionTree' (with any cycle number)

    Available keys: (none!)

in file /home/aegis/ether/Research_HEP/Dataset_ver4/Data/reduce_root/A/run299055/root_299055_26.root
Error reading root_299055_54.root: not found: 'CollectionTree' (with any cycle number)

    Available keys: (none!)

in file /home/aegis/ether/Research_H

In [5]:
print("Total Luminosity being processed:", total_lumi)

Total Luminosity being processed: 25.289004522843967


### Process Data

In [3]:
# === CONFIGURATION ===
BASE_PATH = "/home/aegis/ether/Research_HEP/Dataset_ver4/Data/reduce_root" # Updated to ver4 from your first script
OUTPUT_DIR = "Regions_data_ver3"
BATCH_SIZE = "500 MB"
MAX_WORKERS = 3

# === BRANCHES ===
BRANCHES = [
    # --- MET ---
    "MET_Core_AnalysisMETAuxDyn_mpx", 
    "MET_Core_AnalysisMETAuxDyn_mpy", 
    "MET_Core_AnalysisMETAuxDyn_sumet",
    "AnalysisJetsAuxDyn_NumTrkPt500",
    
    # --- Small-R Jets ---
    "AnalysisJetsAuxDyn_pt", 
    "AnalysisJetsAuxDyn_eta", 
    "AnalysisJetsAuxDyn_phi", 
    "AnalysisJetsAuxDyn_NNJvtPass",
    
    # --- Large-R Jets ---
    "AnalysisLargeRJetsAuxDyn_pt", 
    "AnalysisLargeRJetsAuxDyn_eta", 
    "AnalysisLargeRJetsAuxDyn_phi", 
    "AnalysisLargeRJetsAuxDyn_m", 
    "AnalysisLargeRJetsAuxDyn_Tau1_wta", 
    "AnalysisLargeRJetsAuxDyn_Tau2_wta", 
    "AnalysisLargeRJetsAuxDyn_Tau3_wta", 
    
    # --- Electrons (pt & phi added for MET recalculation) ---
    "AnalysisElectronsAuxDyn_DFCommonElectronsLHTight", 
    "AnalysisElectronsAuxDyn_pt", 
    "AnalysisElectronsAuxDyn_phi",
    "AnalysisElectronsAuxDyn_eta",
    
    # --- Muons (pt, eta, phi, & charge added for 2L CR Z-mass cut) ---
    "AnalysisMuonsAuxDyn_muonType", 
    "AnalysisMuonsAuxDyn_quality", 
    "AnalysisMuonsAuxDyn_pt", 
    "AnalysisMuonsAuxDyn_eta", 
    "AnalysisMuonsAuxDyn_phi", 
    
    # --- Taus ---
    "AnalysisTauJetsAuxDyn_JetDeepSetTight",
    
    # --- B-tagging ---
    "BTagging_AntiKt4EMPFlowAuxDyn_DL1dv01_pu", 
    "BTagging_AntiKt4EMPFlowAuxDyn_DL1dv01_pc", 
    "BTagging_AntiKt4EMPFlowAuxDyn_DL1dv01_pb"
]

In [4]:
# === 1. CUTFLOW TRACKER CLASS ===
class CutflowTracker:
    def __init__(self):
        self.steps = []
        self.raw_counts = {}
        self.weighted_counts = {}

    def update(self, step_name, events):
        if step_name not in self.steps: self.steps.append(step_name)
        n_raw = len(events)
        w_sum = ak.sum(np.ones(len(events))) if len(events) > 0 else 0.0
        self.raw_counts[step_name] = self.raw_counts.get(step_name, 0) + n_raw
        self.weighted_counts[step_name] = self.weighted_counts.get(step_name, 0.0) + w_sum

    def save_csv(self, process_name, output_dir):
        data = []
        if not self.steps: return
        initial_w = self.weighted_counts.get(self.steps[0], 1.0)
        if initial_w == 0: initial_w = 1.0 
        prev_w = initial_w

        for step in self.steps:
            raw = self.raw_counts[step]
            weighted = self.weighted_counts[step]
            abs_eff = (weighted / initial_w) * 100
            rel_eff = (weighted / prev_w) * 100 if prev_w > 0 else 0
            
            data.append({
                "Step": step, "Raw Events": raw, "Weighted Yield": weighted,
                "Absolute Eff (%)": f"{abs_eff:.2f}", "Relative Eff (%)": f"{rel_eff:.2f}"
            })
            prev_w = weighted
            
        df = pd.DataFrame(data)
        os.makedirs(output_dir, exist_ok=True)
        df.to_csv(os.path.join(output_dir, f"cutflow_{process_name}.csv"), index=False)
        print(f"Saved cutflow for {process_name}")
def delta_r(eta1, phi1, eta2, phi2):
    """Calculates angular distance ΔR with proper phi wrapping."""
    deta = eta1 - eta2
    dphi = np.abs(phi1 - phi2)
    dphi = ak.where(dphi > np.pi, 2 * np.pi - dphi, dphi)
    return np.sqrt(deta**2 + dphi**2)

def apply_overlap_removal(events):
    """
    Applies overlap removal criteria using purely flat parallel arrays.
    """
    jet_pt  = events["valid_jet_pt"]
    jet_eta = events["valid_jet_eta"]
    jet_phi = events["valid_jet_phi"]
    jet_trk = events["valid_jet_trk"] 

    ele_pt  = events["valid_elec_pt"]
    ele_eta = events["valid_elec_eta"]
    ele_phi = events["valid_elec_phi"]

    mu_pt   = events["valid_muon_pt"]
    mu_eta  = events["valid_muon_eta"]
    mu_phi  = events["valid_muon_phi"]

    # 1. JET-ELECTRON OR: Remove jet if ΔR(jet, e) < 0.2
    je_eta_j, je_eta_e = ak.unzip(ak.cartesian((jet_eta, ele_eta), nested=True))
    je_phi_j, je_phi_e = ak.unzip(ak.cartesian((jet_phi, ele_phi), nested=True))
    
    dR_je = delta_r(je_eta_j, je_phi_j, je_eta_e, je_phi_e)
    jet_pass_e = ak.all(dR_je >= 0.2, axis=-1)
    
    jet_pt  = jet_pt[jet_pass_e]
    jet_eta = jet_eta[jet_pass_e]
    jet_phi = jet_phi[jet_pass_e]
    jet_trk = jet_trk[jet_pass_e]

    # 2. JET-MUON OR: Remove jet if ΔR(jet, mu) < 0.4 AND jet tracks < 3
    jm_eta_j, jm_eta_m = ak.unzip(ak.cartesian((jet_eta, mu_eta), nested=True))
    jm_phi_j, jm_phi_m = ak.unzip(ak.cartesian((jet_phi, mu_phi), nested=True))
    
    dR_jm = delta_r(jm_eta_j, jm_phi_j, jm_eta_m, jm_phi_m)
    jet_near_mu = ak.any(dR_jm < 0.4, axis=-1)
    
    jet_pass_mu = ~(jet_near_mu & (jet_trk < 3))
    
    surv_jet_pt  = jet_pt[jet_pass_mu]
    surv_jet_eta = jet_eta[jet_pass_mu]
    surv_jet_phi = jet_phi[jet_pass_mu]

    # 3. ELECTRON-JET OR: Remove e if ΔR(e, surviving_jet) < 0.4
    ej_eta_e, ej_eta_j = ak.unzip(ak.cartesian((ele_eta, surv_jet_eta), nested=True))
    ej_phi_e, ej_phi_j = ak.unzip(ak.cartesian((ele_phi, surv_jet_phi), nested=True))
    
    dR_ej = delta_r(ej_eta_e, ej_phi_e, ej_eta_j, ej_phi_j)
    ele_pass_j = ak.all(dR_ej >= 0.4, axis=-1)
    
    surv_ele_pt  = ele_pt[ele_pass_j]
    surv_ele_eta = ele_eta[ele_pass_j]
    surv_ele_phi = ele_phi[ele_pass_j]

    # 4. MUON-JET OR: Remove mu if ΔR(mu, surviving_jet) < 0.4
    mj_eta_m, mj_eta_j = ak.unzip(ak.cartesian((mu_eta, surv_jet_eta), nested=True))
    mj_phi_m, mj_phi_j = ak.unzip(ak.cartesian((mu_phi, surv_jet_phi), nested=True))
    
    dR_mj = delta_r(mj_eta_m, mj_phi_m, mj_eta_j, mj_phi_j)
    mu_pass_j = ak.all(dR_mj >= 0.4, axis=-1)
    
    surv_mu_pt  = mu_pt[mu_pass_j]
    surv_mu_eta = mu_eta[mu_pass_j]
    surv_mu_phi = mu_phi[mu_pass_j]

    # 5. Save the cleaned arrays back to the event
    events["post_OR_jet_pt"]  = surv_jet_pt
    events["post_OR_jet_eta"] = surv_jet_eta
    events["post_OR_jet_phi"] = surv_jet_phi

    events["post_OR_elec_pt"]  = surv_ele_pt
    events["post_OR_elec_eta"] = surv_ele_eta
    events["post_OR_elec_phi"] = surv_ele_phi

    events["post_OR_muon_pt"]  = surv_mu_pt
    events["post_OR_muon_eta"] = surv_mu_eta
    events["post_OR_muon_phi"] = surv_mu_phi

    return events
# === 2. EVENT CLEANING (PRESELECTION) ===
def apply_preselection(events, tracker):
    """
    Applies cuts and updates the tracker. Returns filtered events.
    Incorporates MET calculation based on CERN-EP-2023-084.
    """
    if len(events) == 0: return events
    tracker.update("Initial Raw Batch", events)

    # ====================================================================
    # 1. KINEMATIC SELECTION
    # ====================================================================
    raw_jet_pt  = events["AnalysisJetsAuxDyn_pt"] / 1000.0
    raw_jet_eta = events["AnalysisJetsAuxDyn_eta"]
    raw_jet_phi = events["AnalysisJetsAuxDyn_phi"]
    raw_jet_jvt = events["AnalysisJetsAuxDyn_NNJvtPass"]
    
    # --- THE FIX FOR NumTrkPt500 ---
    raw_jet_trk_3d = events["AnalysisJetsAuxDyn_NumTrkPt500"]
    
    # ak.firsts safely grabs the 0th element (Primary Vertex) of the 3rd dimension.
    # If the list is empty, it returns None. ak.fill_none replaces those Nones with 0.
    raw_jet_trk = ak.fill_none(ak.firsts(raw_jet_trk_3d, axis=2), 0)
    # -------------------------------

    raw_jet_pb  = events["BTagging_AntiKt4EMPFlowAuxDyn_DL1dv01_pb"]
    raw_jet_pc  = events["BTagging_AntiKt4EMPFlowAuxDyn_DL1dv01_pc"]
    raw_jet_pu  = events["BTagging_AntiKt4EMPFlowAuxDyn_DL1dv01_pu"]

    # Acceptance mask (|eta| < 2.8)
    eta_mask = np.abs(raw_jet_eta) < 2.8
    
    events["valid_jet_pt"]  = raw_jet_pt[eta_mask]
    events["valid_jet_eta"] = raw_jet_eta[eta_mask]
    events["valid_jet_phi"] = raw_jet_phi[eta_mask]
    events["valid_jet_jvt"] = raw_jet_jvt[eta_mask]
    events["valid_jet_trk"] = raw_jet_trk[eta_mask]  # This will now work perfectly!
    events["valid_jet_pb"]  = raw_jet_pb[eta_mask]
    events["valid_jet_pc"]  = raw_jet_pc[eta_mask]
    events["valid_jet_pu"]  = raw_jet_pu[eta_mask]

    # Require at least 2 jets in acceptance
    has_at_least_2 = ak.num(events["valid_jet_pt"]) >= 2
    events = events[has_at_least_2]
    if len(events) == 0: return events

    # Kinematic cuts (Leading > 250, Subleading > 30)
    leading_pt = events["valid_jet_pt"][:, 0]
    subleading_pt = events["valid_jet_pt"][:, 1]
    kin_mask = (leading_pt > 250) & (subleading_pt > 30)
    events = events[kin_mask]
    
    # Remove jets under 30 pT inside the surviving events
    pt30_mask = events["valid_jet_pt"] > 30
    jet_branches = ["pt", "eta", "phi", "jvt", "trk", "pb", "pc", "pu"]
    
    for b in jet_branches:
        events[f"valid_jet_{b}"] = events[f"valid_jet_{b}"][pt30_mask]

    tracker.update("Jet Kinematic Selection", events)
    if len(events) == 0: return events

    # ====================================================================
    # 2. JVT CLEANING & RE-EVALUATION
    # ====================================================================
    pass_jvt_cut = (events["valid_jet_pt"] >= 60) | (events["valid_jet_jvt"] == 1)
    
    # Update valid jets in-place to drop JVT failures
    for b in jet_branches:
        events[f"valid_jet_{b}"] = events[f"valid_jet_{b}"][pass_jvt_cut]
    
    has_at_least_2_clean = ak.num(events["valid_jet_pt"]) >= 2
    events = events[has_at_least_2_clean]
    if len(events) == 0: return events

    leading_clean_pass = events["valid_jet_pt"][:, 0] > 250
    events = events[leading_clean_pass]
    
    tracker.update("JVT Cleaning & Re-evaluation", events)
    if len(events) == 0: return events

    # ====================================================================
    # 3. B-JET VETO
    # ====================================================================
    fc = 0.080
    with np.errstate(divide='ignore', invalid='ignore'):
        dl1_score = np.log(events["valid_jet_pb"] / (fc * events["valid_jet_pc"] + (1 - fc) * events["valid_jet_pu"] + 1e-10))

    n_bjets = ak.sum(dl1_score > 1.42, axis=1)
    events["n_bjets"] = n_bjets
    events = events[n_bjets <= 1]

    tracker.update("B-jet Veto", events)
    if len(events) == 0: return events

    # ====================================================================
    # 4. LARGE-R JET REQUIREMENT
    # ====================================================================
    ljet_pt = events["AnalysisLargeRJetsAuxDyn_pt"] / 1000.0
    events = events[ak.num(ljet_pt) >= 2]
    
    tracker.update("Large-R Jet >= 2", events)
    if len(events) == 0: return events

    # ====================================================================
    # 5. ELECTRON & MUON DEFINITION
    # ====================================================================
    ele_pt  = events["AnalysisElectronsAuxDyn_pt"] / 1000.0
    ele_eta = events["AnalysisElectronsAuxDyn_eta"]
    ele_phi = events["AnalysisElectronsAuxDyn_phi"]
    ele_id  = events["AnalysisElectronsAuxDyn_DFCommonElectronsLHTight"]

    ele_eta_abs = np.abs(ele_eta)
    ele_mask = (ele_pt > 7) & (ele_id == 1) & (ele_eta_abs < 2.5) & ~((ele_eta_abs > 1.37) & (ele_eta_abs < 1.52))

    events["valid_elec_pt"]  = ele_pt[ele_mask]
    events["valid_elec_eta"] = ele_eta[ele_mask]
    events["valid_elec_phi"] = ele_phi[ele_mask]

    mu_pt   = events["AnalysisMuonsAuxDyn_pt"] / 1000.0
    mu_eta  = events["AnalysisMuonsAuxDyn_eta"]
    mu_phi  = events["AnalysisMuonsAuxDyn_phi"]
    mu_qual = events["AnalysisMuonsAuxDyn_quality"]
    mu_type = events["AnalysisMuonsAuxDyn_muonType"]

    mu_mask = (mu_pt > 7) & (np.abs(mu_eta) < 2.5) & ((mu_qual == 8) | (mu_qual == 9)) & (mu_type == 0)

    events["valid_muon_pt"]  = mu_pt[mu_mask]
    events["valid_muon_eta"] = mu_eta[mu_mask]
    events["valid_muon_phi"] = mu_phi[mu_mask]

    tracker.update("Lepton Definition", events)

    # ====================================================================
    # 6. OVERLAP REMOVAL & EVENT RE-EVALUATION
    # ====================================================================
    events = apply_overlap_removal(events)
    
    has_at_least_2_postOR = ak.num(events["post_OR_jet_pt"]) >= 2
    events = events[has_at_least_2_postOR]
    if len(events) == 0: return events

    leading_postOR_pass = events["post_OR_jet_pt"][:, 0] > 250
    events = events[leading_postOR_pass]

    tracker.update("Post-OR Jet Requirements", events)
    if len(events) == 0: return events

    # ====================================================================
    # 7. MET REBUILDING (Excluding Muons)
    # ====================================================================
    soft_px = events["MET_Core_AnalysisMETAuxDyn_mpx"][:, 0] / 1000.0
    soft_py = events["MET_Core_AnalysisMETAuxDyn_mpy"][:, 0] / 1000.0

    sum_j_px = ak.sum(events["post_OR_jet_pt"] * np.cos(events["post_OR_jet_phi"]), axis=1)
    sum_j_py = ak.sum(events["post_OR_jet_pt"] * np.sin(events["post_OR_jet_phi"]), axis=1)

    sum_e_px = ak.sum(events["post_OR_elec_pt"] * np.cos(events["post_OR_elec_phi"]), axis=1)
    sum_e_py = ak.sum(events["post_OR_elec_pt"] * np.sin(events["post_OR_elec_phi"]), axis=1)

    met_px = -(sum_j_px + sum_e_px + soft_px)
    met_py = -(sum_j_py + sum_e_py + soft_py)

    events["met_recalc_pt"]  = np.sqrt(met_px**2 + met_py**2)
    events["met_recalc_phi"] = np.arctan2(met_py, met_px)

    tracker.update("MET Rebuilding", events)
    if len(events) == 0: return events

    # ====================================================================
    # 8. dPHI(JET, MET) REQUIREMENT
    # ====================================================================
    dphi = np.abs(events["post_OR_jet_phi"] - events["met_recalc_phi"])
    dphi = ak.where(dphi > np.pi, 2 * np.pi - dphi, dphi)

    pass_dphi = ak.any(dphi < 2.0, axis=1)
    events = events[pass_dphi]

    tracker.update("dPhi(Jet, MET) < 2.0", events)
    if len(events) == 0: return events

    # ====================================================================
    # 9. FINAL COUNTS & TAU VETO
    # ====================================================================
    events["ht"]    = ak.sum(events["post_OR_jet_pt"], axis=1)
    events["n_mu"]  = ak.num(events["post_OR_muon_pt"])
    events["n_ele"] = ak.num(events["post_OR_elec_pt"])
    events["n_tau"] = ak.sum(events["AnalysisTauJetsAuxDyn_JetDeepSetTight"] == 1, axis=1)
    
    events = events[events["n_tau"] == 0]

    tracker.update("Tau Veto / Preselection Complete", events)
    
    return events


# === 3. REGION SPLITTING & SAVING ===
def split_and_save(events, process_name, tracker, batch_suffix):
    if len(events) == 0: return

    # Global kinematic masks
    kin_mask_sr = (events["met_recalc_pt"] > 600) & (events["ht"] > 600)
    kin_mask_cr = (events["met_recalc_pt"] > 250) & (events["met_recalc_pt"] <= 600) & (events["ht"] <= 600)
    kin_mask_qcd = (events["met_recalc_pt"] <= 600) & (events["ht"] <= 600)

    # ====================================================================
    # 1. ISOLATED Z-MASS CALCULATION FOR 2L CR
    # ====================================================================
    # First, get the events that pass the base 2L requirements
    base_2L_mask = kin_mask_cr & (events["n_mu"] == 2) & (events["n_bjets"] == 0)
    events_2L = events[base_2L_mask]

    if len(events_2L) > 0:
        # Extract the flat arrays for the leading (0) and subleading (1) muons
        mu1_pt  = events_2L["post_OR_muon_pt"][:, 0]
        mu2_pt  = events_2L["post_OR_muon_pt"][:, 1]
        
        mu1_eta = events_2L["post_OR_muon_eta"][:, 0]
        mu2_eta = events_2L["post_OR_muon_eta"][:, 1]
        
        mu1_phi = events_2L["post_OR_muon_phi"][:, 0]
        mu2_phi = events_2L["post_OR_muon_phi"][:, 1]
        
        deta = mu1_eta - mu2_eta
        dphi = np.abs(mu1_phi - mu2_phi)
        dphi = ak.where(dphi > np.pi, 2 * np.pi - dphi, dphi) # Handle wrap-around
        
        # Calculate invariant mass
        mass_mumu = np.sqrt(2 * mu1_pt * mu2_pt * (np.cosh(deta) - np.cos(dphi)))
        
        # Apply the Z-mass window (66 to 116 GeV)
        events_2L_final = events_2L[(mass_mumu >= 66) & (mass_mumu <= 116)]
    else:
        # Failsafe if the batch had no 2L events
        events_2L_final = events_2L

    regions = {
        "SR": events[kin_mask_sr & (events["n_mu"] == 0) & (events["n_bjets"] <= 1)],
        
        # 1L CR (W+jets dominated): 1 muon, 0 b-jets
        "CR_1L": events[kin_mask_cr & (events["n_mu"] == 1) & (events["n_bjets"] == 0)],
        
        # 1L1B CR (Top dominated): 1 muon, exactly 1 b-jet
        "CR_1L1B": events[kin_mask_cr & (events["n_mu"] == 1) & (events["n_bjets"] == 1)],

        # 2L CR (Z+jets dominated): 2 muons, 0 b-jets + Invariant Mass Cut
        "CR_2L": events_2L_final,

        # Multijet Reweighting Region (MJRR)
        "CR_MJRR": events[kin_mask_qcd & (events["n_mu"] == 0)]
    }

    for reg, data in regions.items():
        tracker.update(f"Region: {reg}", data)
        if len(data) > 0:
            save_path = os.path.join(OUTPUT_DIR, reg)
            os.makedirs(save_path, exist_ok=True)
            file_name = os.path.join(save_path, f"{process_name}{batch_suffix}.parquet")
            ak.to_parquet(data, file_name)

# === 4. MAIN PIPELINE ===
def process_full_dataset(process_name):
    print(f"--> Processing {process_name}...")
    tracker = CutflowTracker()
    batch_counter = 0 

    subprocesses = os.listdir(os.path.join(BASE_PATH, process_name))
    # print(subprocesses)
    for subp in subprocesses:
        # --- THE GRL CHECK ---
        try:
            run_num = int(subp.replace("run", ""))
            if run_num not in GRL_SET:
                continue  # Skip this directory entirely!
        except ValueError:
            continue  # Skip if the folder isn't named with a run number
        # ---------------------
        ttbar_path = os.path.join(BASE_PATH, process_name)
        subp_dir = os.path.join(ttbar_path, f"{subp}")
        
        if not os.path.exists(subp_dir): continue
        files = sorted([os.path.join(subp_dir, f) for f in os.listdir(subp_dir) if f.endswith(".root")])
        if not files: continue
        
        for file_path in files:
            try:
                # uproot.iterate now processes just this one file in batches
                for events in uproot.iterate(
                    f"{file_path}:CollectionTree", 
                    BRANCHES, 
                    library="ak", 
                    step_size=BATCH_SIZE
                ):
                    # Clean, Split, and Save as normal
                    cleaned_events = apply_preselection(events, tracker)
                    split_and_save(cleaned_events, process_name, tracker, f"_batch{batch_counter}")
                    batch_counter += 1
                    
            except Exception as e:
                # If this specific file is corrupted, log it, and the loop naturally 
                # continues to the next file_path in the 'files' list!
                print(f"  [WARNING] Skipping corrupted file {file_path}")
                print(f"  [ERROR DETAILS] {e}")
                continue 

    tracker.save_csv(process_name, OUTPUT_DIR)
    print(f"<-- Finished {process_name}")

In [5]:
processes = ["A", "B", "C", "D", "E", "F", "G", "I", "K", "L"] 
    
start_time = time.time()
now = datetime.datetime.now()
cpu_cores = multiprocessing.cpu_count()

print("="*60)
print(f"  HEP DATA PROCESSING PIPELINE (BATCHED)")
print(f"  Date:       {now.strftime('%Y-%m-%d')}")
print(f"  System:     {cpu_cores} CPU Cores Detected")
print(f"  Config:     {MAX_WORKERS} Workers | Batch Size: {BATCH_SIZE}")
print("="*60)

# Using ProcessPoolExecutor to bypass the GIL
with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
    executor.map(process_full_dataset, processes)

elapsed = time.time() - start_time
print("="*60)
print(f"  PIPELINE COMPLETE")
print(f"  Total Duration: {elapsed/60:.2f} minutes")
print("="*60)

  HEP DATA PROCESSING PIPELINE (BATCHED)
  Date:       2026-04-09
  System:     128 CPU Cores Detected
  Config:     3 Workers | Batch Size: 500 MB
--> Processing A...--> Processing B...--> Processing C...


  [WARNING] Skipping corrupted file /home/aegis/ether/Research_HEP/Dataset_ver4/Data/reduce_root/C/run302391/root_302391_36.root
  [ERROR DETAILS] not found: 'CollectionTree' (with any cycle number)

    Available keys: (none!)

in file /home/aegis/ether/Research_HEP/Dataset_ver4/Data/reduce_root/C/run302391/root_302391_36.root
  [WARNING] Skipping corrupted file /home/aegis/ether/Research_HEP/Dataset_ver4/Data/reduce_root/C/run302391/root_302391_37.root
  [ERROR DETAILS] expected Chunk of length 403,
received 0 bytes from FSSpecSource
for file path /home/aegis/ether/Research_HEP/Dataset_ver4/Data/reduce_root/C/run302391/root_302391_37.root
  [WARNING] Skipping corrupted file /home/aegis/ether/Research_HEP/Dataset_ver4/Data/reduce_root/C/run302391/root_302391_38.root
  [ERROR DETAI

### Determine MC Events Scale Factor

In [2]:
import os
import json
import time
import datetime
import multiprocessing
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import awkward as ak
import uproot

In [3]:
MC_BASE_PATH = Path("/home/aegis/ether/Research_HEP/Dataset_ver4/MC/reduce_root")
METADATA_FILE_MC = Path("mc_metadata.json")
LUMINOSITY = 25.289004522843967

with open(METADATA_FILE_MC, "r") as f:
    mc_metadata = json.load(f)

def count_actual_events(sub_path: Path) -> int:
    num_events = 0
    sum_weights = 0
    for file in os.listdir(sub_path):
        if file.endswith(".root"): 
            file_path = f"{sub_path}/{file}"
            try:
                tree = uproot.open(f"{file_path}:CollectionTree")
                num_events += tree.num_entries
                
                # Fetch the array, grab the 0th index (nominal weight), and sum it
                weights = tree["EventInfoAuxDyn_mcEventWeights"].array()
                sum_weights += ak.sum(weights[:, 0])
                
            except Exception as e:
                print(f"Error reading {file_path}: {e}")
    return num_events, sum_weights

def analyze_process(mc_process, sf_dict):
    """
    Calculate the scale factor for each subprocesses of the MC process
    """
    print(f"\n--- Analyzing MC process {mc_process} ---")
    print("Available subprocesses: ")
    subprocess_map = {process: list(sub_dict.keys()) for process, sub_dict in mc_metadata.items()}
    print(subprocess_map[mc_process])

    # Check if all these subprocesses exist in my dataset
    process_sub = [sub.replace("mc20_13TeV_MC_", "") for sub in os.listdir(f"{MC_BASE_PATH}/{mc_process}")]
    print(process_sub)

    for sub in process_sub:
        # Check if within the list of sub
        if sub not in subprocess_map[mc_process]:
            print(sub, "not in the list")
            continue

        sub_path = f"{MC_BASE_PATH}/{mc_process}/mc20_13TeV_MC_{sub}"

        num_events, sum_of_weights = count_actual_events(sub_path)
        

        sum_weights_global = mc_metadata[mc_process][sub]["sum_w"]
        print("="*50)
        print(f"{sub} has {num_events} total events")
        
        print("Calculating Scale Factor...")
        # get cross section
        xsec_pb = mc_metadata[mc_process][sub]["xsec_pb"]
        print(f"{sub} has a {xsec_pb} pb cross section")
        xsec_fb = xsec_pb *1000
        print(f"Cross section in fb: {xsec_fb}")
        scale_factor_ngen = xsec_fb*LUMINOSITY/num_events
        scale_factor_sub_weight = xsec_fb*LUMINOSITY/sum_of_weights
        scale_factor_tot_weight = xsec_fb*LUMINOSITY/sum_weights_global
        print(f"Subprocess {sub} has {scale_factor_ngen} scale factor, using N generated method")
        print(f"Subprocess {sub} has {scale_factor_sub_weight} scale factor, using partial sum of weights method")
        print(f"Subprocess {sub} has {scale_factor_tot_weight} scale factor, using global sum of weights method")
        sf_dict[sub] = {
            "weight_ngen": scale_factor_ngen,
            "weight_sub_weight": scale_factor_sub_weight,
            "weight_tot_weight": scale_factor_tot_weight
        }
        print()


In [4]:
scale_factor_dict = {}

mc_processes = "Diboson", "Multijet", "Single_top", "ttbar", "Wjets", "Zjets"
for mc_proc in mc_processes:
    analyze_process(mc_proc, scale_factor_dict)


--- Analyzing MC process Diboson ---
Available subprocesses: 
['Sh_2211_WlvWqq', 'Sh_2211_WlvZqq', 'Sh_2211_WqqZvv', 'Sh_2211_ZqqZvv']
['Sh_2211_WlvZqq', 'Sh_2211_WlvWqq', 'Sh_2211_WqqZvv', 'Sh_2211_ZqqZvv']
Sh_2211_WlvZqq has 420000 total events
Calculating Scale Factor...
Sh_2211_WlvZqq has a 8.582 pb cross section
Cross section in fb: 8582.0
Subprocess Sh_2211_WlvZqq has 0.5167386590834451 scale factor, using N generated method
Subprocess Sh_2211_WlvZqq has 1.8817905584000982e-05 scale factor, using partial sum of weights method
Subprocess Sh_2211_WlvZqq has 0.0011867730916629764 scale factor, using global sum of weights method

Sh_2211_WlvWqq has 2241000 total events
Calculating Scale Factor...
Sh_2211_WlvWqq has a 109.28 pb cross section
Cross section in fb: 109280.0
Subprocess Sh_2211_WlvWqq has 1.2331916172496158 scale factor, using N generated method
Subprocess Sh_2211_WlvWqq has 4.43123963123071e-06 scale factor, using partial sum of weights method
Subprocess Sh_2211_WlvWqq h

In [5]:
scale_factor_dict

{'Sh_2211_WlvZqq': {'weight_ngen': 0.5167386590834451,
  'weight_sub_weight': np.float32(1.8817906e-05),
  'weight_tot_weight': 0.0011867730916629764},
 'Sh_2211_WlvWqq': {'weight_ngen': 1.2331916172496158,
  'weight_sub_weight': np.float32(4.4312396e-06),
  'weight_tot_weight': 0.0002810798424014236},
 'Sh_2211_WqqZvv': {'weight_ngen': 0.40456224105860167,
  'weight_sub_weight': np.float32(1.9262789e-05),
  'weight_tot_weight': 0.0014905842708600148},
 'Sh_2211_ZqqZvv': {'weight_ngen': 0.7044673693246901,
  'weight_sub_weight': np.float32(2.1895896e-05),
  'weight_tot_weight': 2.05595862401898e-10},
 'Pythia8EvtGen_A14NNPDF23LO_jetjet_JZ9WithSW': {'weight_ngen': 0.008277512663735545,
  'weight_sub_weight': np.float32(96098790.0),
  'weight_tot_weight': 96063976.75515139},
 'Pythia8EvtGen_A14NNPDF23LO_jetjet_JZ4WithSW': {'weight_ngen': 209.37933928073954,
  'weight_sub_weight': np.float32(939166140.0),
  'weight_tot_weight': 576523679.7406839},
 'Pythia8EvtGen_A14NNPDF23LO_jetjet_JZ11W

### Process MC

In [6]:
# === CONFIGURATION ===
BASE_PATH = "/home/aegis/ether/Research_HEP/Dataset_ver4/MC/reduce_root"
OUTPUT_DIR = "Regions_MC_ver2"
BATCH_SIZE = "500 MB"
MAX_WORKERS = 3

# === BRANCHES ===
BRANCHES = [
    "EventInfoAuxDyn_mcEventWeights",
    # --- MET ---
    "MET_Core_AnalysisMETAuxDyn_mpx", 
    "MET_Core_AnalysisMETAuxDyn_mpy", 
    "MET_Core_AnalysisMETAuxDyn_sumet",
    "AnalysisJetsAuxDyn_NumTrkPt500",
    
    # --- Small-R Jets ---
    "AnalysisJetsAuxDyn_pt", 
    "AnalysisJetsAuxDyn_eta", 
    "AnalysisJetsAuxDyn_phi", 
    "AnalysisJetsAuxDyn_NNJvtPass",
    
    # --- Large-R Jets ---
    "AnalysisLargeRJetsAuxDyn_pt", 
    "AnalysisLargeRJetsAuxDyn_eta", 
    "AnalysisLargeRJetsAuxDyn_phi", 
    "AnalysisLargeRJetsAuxDyn_m", 
    "AnalysisLargeRJetsAuxDyn_Tau1_wta", 
    "AnalysisLargeRJetsAuxDyn_Tau2_wta", 
    "AnalysisLargeRJetsAuxDyn_Tau3_wta", 
    
    # --- Electrons (pt & phi added for MET recalculation) ---
    "AnalysisElectronsAuxDyn_DFCommonElectronsLHTight", 
    "AnalysisElectronsAuxDyn_pt", 
    "AnalysisElectronsAuxDyn_phi",
    "AnalysisElectronsAuxDyn_eta",
    
    # --- Muons (pt, eta, phi, & charge added for 2L CR Z-mass cut) ---
    "AnalysisMuonsAuxDyn_muonType", 
    "AnalysisMuonsAuxDyn_quality", 
    "AnalysisMuonsAuxDyn_pt", 
    "AnalysisMuonsAuxDyn_eta", 
    "AnalysisMuonsAuxDyn_phi", 
    
    # --- Taus ---
    "AnalysisTauJetsAuxDyn_JetDeepSetTight",
    
    # --- B-tagging ---
    "BTagging_AntiKt4EMPFlowAuxDyn_DL1dv01_pu", 
    "BTagging_AntiKt4EMPFlowAuxDyn_DL1dv01_pc", 
    "BTagging_AntiKt4EMPFlowAuxDyn_DL1dv01_pb"
]

In [ ]:
# === 1. CUTFLOW TRACKER CLASS ===
class CutflowTracker:
    def __init__(self):
        self.steps = []
        self.raw_counts = {}
        # Track all three weight variants
        self.weighted_counts_ngen = {}
        self.weighted_counts_sub = {}
        self.weighted_counts_tot = {}


    def update(self, step_name, events):
        if step_name not in self.steps: self.steps.append(step_name)
        n_raw = len(events)
        
        # Safely sum each weight array (if array is empty, default to 0.0)
        w_ngen = ak.sum(events["weight_ngen"]) if n_raw > 0 else 0.0
        w_sub = ak.sum(events["weight_sub_weight"]) if n_raw > 0 else 0.0
        w_tot = ak.sum(events["weight_tot_weight"]) if n_raw > 0 else 0.0
        
        self.raw_counts[step_name] = self.raw_counts.get(step_name, 0) + n_raw
        self.weighted_counts_ngen[step_name] = self.weighted_counts_ngen.get(step_name, 0.0) + w_ngen
        self.weighted_counts_sub[step_name] = self.weighted_counts_sub.get(step_name, 0.0) + w_sub
        self.weighted_counts_tot[step_name] = self.weighted_counts_tot.get(step_name, 0.0) + w_tot

    def save_csv(self, process_name, output_dir):
        data = []
        if not self.steps: return
        
        # We calculate efficiencies based on the mathematically correct "Global" weight
        initial_w_tot = self.weighted_counts_tot.get(self.steps[0], 1.0)
        if initial_w_tot == 0: initial_w_tot = 1.0 
        prev_w_tot = initial_w_tot

        for step in self.steps:
            raw = self.raw_counts[step]
            w_ngen = self.weighted_counts_ngen[step]
            w_sub = self.weighted_counts_sub[step]
            w_tot = self.weighted_counts_tot[step]
            
            # Efficiency calculations using the Global weight
            abs_eff = (w_tot / initial_w_tot) * 100
            rel_eff = (w_tot / prev_w_tot) * 100 if prev_w_tot > 0 else 0
            
            data.append({
                "Step": step, 
                "Raw Events": raw, 
                "Yield (Global SumW)": w_tot,         # The correct physics yield
                "Yield (Partial SumW)": w_sub,        # The subset trap
                "Yield (N Generated)": w_ngen,        # The LO trap
                "Absolute Eff (%)": f"{abs_eff:.2f}", 
                "Relative Eff (%)": f"{rel_eff:.2f}"
            })
            prev_w_tot = w_tot
            
        df = pd.DataFrame(data)
        os.makedirs(output_dir, exist_ok=True)
        df.to_csv(os.path.join(output_dir, f"cutflow_{process_name}.csv"), index=False)
        print(f"Saved cutflow for {process_name}")


# ====================================================================
# HELPER FUNCTIONS
# ====================================================================
def delta_r(eta1, phi1, eta2, phi2):
    """Calculates angular distance ΔR with proper phi wrapping."""
    deta = eta1 - eta2
    dphi = np.abs(phi1 - phi2)
    dphi = ak.where(dphi > np.pi, 2 * np.pi - dphi, dphi)
    return np.sqrt(deta**2 + dphi**2)

def apply_overlap_removal(events):
    """
    Applies overlap removal criteria using purely flat parallel arrays.
    """
    jet_pt  = events["valid_jet_pt"]
    jet_eta = events["valid_jet_eta"]
    jet_phi = events["valid_jet_phi"]
    jet_trk = events["valid_jet_trk"] 

    ele_pt  = events["valid_elec_pt"]
    ele_eta = events["valid_elec_eta"]
    ele_phi = events["valid_elec_phi"]

    mu_pt   = events["valid_muon_pt"]
    mu_eta  = events["valid_muon_eta"]
    mu_phi  = events["valid_muon_phi"]

    # 1. JET-ELECTRON OR: Remove jet if ΔR(jet, e) < 0.2
    je_eta_j, je_eta_e = ak.unzip(ak.cartesian((jet_eta, ele_eta), nested=True))
    je_phi_j, je_phi_e = ak.unzip(ak.cartesian((jet_phi, ele_phi), nested=True))
    
    dR_je = delta_r(je_eta_j, je_phi_j, je_eta_e, je_phi_e)
    jet_pass_e = ak.all(dR_je >= 0.2, axis=-1)
    
    jet_pt  = jet_pt[jet_pass_e]
    jet_eta = jet_eta[jet_pass_e]
    jet_phi = jet_phi[jet_pass_e]
    jet_trk = jet_trk[jet_pass_e]

    # 2. JET-MUON OR: Remove jet if ΔR(jet, mu) < 0.4 AND jet tracks < 3
    jm_eta_j, jm_eta_m = ak.unzip(ak.cartesian((jet_eta, mu_eta), nested=True))
    jm_phi_j, jm_phi_m = ak.unzip(ak.cartesian((jet_phi, mu_phi), nested=True))
    
    dR_jm = delta_r(jm_eta_j, jm_phi_j, jm_eta_m, jm_phi_m)
    jet_near_mu = ak.any(dR_jm < 0.4, axis=-1)
    
    jet_pass_mu = ~(jet_near_mu & (jet_trk < 3))
    
    surv_jet_pt  = jet_pt[jet_pass_mu]
    surv_jet_eta = jet_eta[jet_pass_mu]
    surv_jet_phi = jet_phi[jet_pass_mu]

    # 3. ELECTRON-JET OR: Remove e if ΔR(e, surviving_jet) < 0.4
    ej_eta_e, ej_eta_j = ak.unzip(ak.cartesian((ele_eta, surv_jet_eta), nested=True))
    ej_phi_e, ej_phi_j = ak.unzip(ak.cartesian((ele_phi, surv_jet_phi), nested=True))
    
    dR_ej = delta_r(ej_eta_e, ej_phi_e, ej_eta_j, ej_phi_j)
    ele_pass_j = ak.all(dR_ej >= 0.4, axis=-1)
    
    surv_ele_pt  = ele_pt[ele_pass_j]
    surv_ele_eta = ele_eta[ele_pass_j]
    surv_ele_phi = ele_phi[ele_pass_j]

    # 4. MUON-JET OR: Remove mu if ΔR(mu, surviving_jet) < 0.4
    mj_eta_m, mj_eta_j = ak.unzip(ak.cartesian((mu_eta, surv_jet_eta), nested=True))
    mj_phi_m, mj_phi_j = ak.unzip(ak.cartesian((mu_phi, surv_jet_phi), nested=True))
    
    dR_mj = delta_r(mj_eta_m, mj_phi_m, mj_eta_j, mj_phi_j)
    mu_pass_j = ak.all(dR_mj >= 0.4, axis=-1)
    
    surv_mu_pt  = mu_pt[mu_pass_j]
    surv_mu_eta = mu_eta[mu_pass_j]
    surv_mu_phi = mu_phi[mu_pass_j]

    # 5. Save the cleaned arrays back to the event
    events["post_OR_jet_pt"]  = surv_jet_pt
    events["post_OR_jet_eta"] = surv_jet_eta
    events["post_OR_jet_phi"] = surv_jet_phi

    events["post_OR_elec_pt"]  = surv_ele_pt
    events["post_OR_elec_eta"] = surv_ele_eta
    events["post_OR_elec_phi"] = surv_ele_phi

    events["post_OR_muon_pt"]  = surv_mu_pt
    events["post_OR_muon_eta"] = surv_mu_eta
    events["post_OR_muon_phi"] = surv_mu_phi

    return events


# ====================================================================
# MAIN PRESELECTION FUNCTION
# ====================================================================
def apply_preselection_MC(events, tracker, subp_name):
    """
    Applies cuts and updates the tracker. Returns filtered events.
    Incorporates MET calculation based on CERN-EP-2023-084.
    """
    if len(events) == 0: return events
    
    # Event Weights
    events["raw_weights"] = events["EventInfoAuxDyn_mcEventWeights"][:, 0]
    events["weight_ngen"] = scale_factor_dict[subp_name]["weight_ngen"]
    events["weight_sub_weight"] = scale_factor_dict[subp_name]["weight_sub_weight"]
    events["weight_tot_weight"] = scale_factor_dict[subp_name]["weight_tot_weight"]

    tracker.update("Initial Raw Batch", events)

    # ====================================================================
    # 1. KINEMATIC SELECTION
    # ====================================================================
    raw_jet_pt  = events["AnalysisJetsAuxDyn_pt"] / 1000.0
    raw_jet_eta = events["AnalysisJetsAuxDyn_eta"]
    raw_jet_phi = events["AnalysisJetsAuxDyn_phi"]
    raw_jet_jvt = events["AnalysisJetsAuxDyn_NNJvtPass"]
    
    # --- THE FIX FOR NumTrkPt500 ---
    raw_jet_trk_3d = events["AnalysisJetsAuxDyn_NumTrkPt500"]
    
    # ak.firsts safely grabs the 0th element (Primary Vertex) of the 3rd dimension.
    # If the list is empty, it returns None. ak.fill_none replaces those Nones with 0.
    raw_jet_trk = ak.fill_none(ak.firsts(raw_jet_trk_3d, axis=2), 0)
    # -------------------------------

    raw_jet_pb  = events["BTagging_AntiKt4EMPFlowAuxDyn_DL1dv01_pb"]
    raw_jet_pc  = events["BTagging_AntiKt4EMPFlowAuxDyn_DL1dv01_pc"]
    raw_jet_pu  = events["BTagging_AntiKt4EMPFlowAuxDyn_DL1dv01_pu"]

    # Acceptance mask (|eta| < 2.8)
    eta_mask = np.abs(raw_jet_eta) < 2.8
    
    events["valid_jet_pt"]  = raw_jet_pt[eta_mask]
    events["valid_jet_eta"] = raw_jet_eta[eta_mask]
    events["valid_jet_phi"] = raw_jet_phi[eta_mask]
    events["valid_jet_jvt"] = raw_jet_jvt[eta_mask]
    events["valid_jet_trk"] = raw_jet_trk[eta_mask]  # This will now work perfectly!
    events["valid_jet_pb"]  = raw_jet_pb[eta_mask]
    events["valid_jet_pc"]  = raw_jet_pc[eta_mask]
    events["valid_jet_pu"]  = raw_jet_pu[eta_mask]

    # Require at least 2 jets in acceptance
    has_at_least_2 = ak.num(events["valid_jet_pt"]) >= 2
    events = events[has_at_least_2]
    if len(events) == 0: return events

    # Kinematic cuts (Leading > 250, Subleading > 30)
    leading_pt = events["valid_jet_pt"][:, 0]
    subleading_pt = events["valid_jet_pt"][:, 1]
    kin_mask = (leading_pt > 250) & (subleading_pt > 30)
    events = events[kin_mask]
    
    # Remove jets under 30 pT inside the surviving events
    pt30_mask = events["valid_jet_pt"] > 30
    jet_branches = ["pt", "eta", "phi", "jvt", "trk", "pb", "pc", "pu"]
    
    for b in jet_branches:
        events[f"valid_jet_{b}"] = events[f"valid_jet_{b}"][pt30_mask]

    tracker.update("Jet Kinematic Selection", events)
    if len(events) == 0: return events

    # ====================================================================
    # 2. JVT CLEANING & RE-EVALUATION
    # ====================================================================
    pass_jvt_cut = (events["valid_jet_pt"] >= 60) | (events["valid_jet_jvt"] == 1)
    
    # Update valid jets in-place to drop JVT failures
    for b in jet_branches:
        events[f"valid_jet_{b}"] = events[f"valid_jet_{b}"][pass_jvt_cut]
    
    has_at_least_2_clean = ak.num(events["valid_jet_pt"]) >= 2
    events = events[has_at_least_2_clean]
    if len(events) == 0: return events

    leading_clean_pass = events["valid_jet_pt"][:, 0] > 250
    events = events[leading_clean_pass]
    
    tracker.update("JVT Cleaning & Re-evaluation", events)
    if len(events) == 0: return events

    # ====================================================================
    # 3. B-JET VETO
    # ====================================================================
    fc = 0.080
    with np.errstate(divide='ignore', invalid='ignore'):
        dl1_score = np.log(events["valid_jet_pb"] / (fc * events["valid_jet_pc"] + (1 - fc) * events["valid_jet_pu"] + 1e-10))

    n_bjets = ak.sum(dl1_score > 1.42, axis=1)
    events["n_bjets"] = n_bjets
    events = events[n_bjets <= 1]

    tracker.update("B-jet Veto", events)
    if len(events) == 0: return events

    # ====================================================================
    # 4. LARGE-R JET REQUIREMENT
    # ====================================================================
    ljet_pt = events["AnalysisLargeRJetsAuxDyn_pt"] / 1000.0
    events = events[ak.num(ljet_pt) >= 2]
    
    tracker.update("Large-R Jet >= 2", events)
    if len(events) == 0: return events

    # ====================================================================
    # 5. ELECTRON & MUON DEFINITION
    # ====================================================================
    ele_pt  = events["AnalysisElectronsAuxDyn_pt"] / 1000.0
    ele_eta = events["AnalysisElectronsAuxDyn_eta"]
    ele_phi = events["AnalysisElectronsAuxDyn_phi"]
    ele_id  = events["AnalysisElectronsAuxDyn_DFCommonElectronsLHTight"]

    ele_eta_abs = np.abs(ele_eta)
    ele_mask = (ele_pt > 7) & (ele_id == 1) & (ele_eta_abs < 2.5) & ~((ele_eta_abs > 1.37) & (ele_eta_abs < 1.52))

    events["valid_elec_pt"]  = ele_pt[ele_mask]
    events["valid_elec_eta"] = ele_eta[ele_mask]
    events["valid_elec_phi"] = ele_phi[ele_mask]

    mu_pt   = events["AnalysisMuonsAuxDyn_pt"] / 1000.0
    mu_eta  = events["AnalysisMuonsAuxDyn_eta"]
    mu_phi  = events["AnalysisMuonsAuxDyn_phi"]
    mu_qual = events["AnalysisMuonsAuxDyn_quality"]
    mu_type = events["AnalysisMuonsAuxDyn_muonType"]

    mu_mask = (mu_pt > 7) & (np.abs(mu_eta) < 2.5) & ((mu_qual == 8) | (mu_qual == 9)) & (mu_type == 0)

    events["valid_muon_pt"]  = mu_pt[mu_mask]
    events["valid_muon_eta"] = mu_eta[mu_mask]
    events["valid_muon_phi"] = mu_phi[mu_mask]

    tracker.update("Lepton Definition", events)

    # ====================================================================
    # 6. OVERLAP REMOVAL & EVENT RE-EVALUATION
    # ====================================================================
    events = apply_overlap_removal(events)
    
    has_at_least_2_postOR = ak.num(events["post_OR_jet_pt"]) >= 2
    events = events[has_at_least_2_postOR]
    if len(events) == 0: return events

    leading_postOR_pass = events["post_OR_jet_pt"][:, 0] > 250
    events = events[leading_postOR_pass]

    tracker.update("Post-OR Jet Requirements", events)
    if len(events) == 0: return events

    # ====================================================================
    # 7. MET REBUILDING (Excluding Muons)
    # ====================================================================
    soft_px = events["MET_Core_AnalysisMETAuxDyn_mpx"][:, 0] / 1000.0
    soft_py = events["MET_Core_AnalysisMETAuxDyn_mpy"][:, 0] / 1000.0

    sum_j_px = ak.sum(events["post_OR_jet_pt"] * np.cos(events["post_OR_jet_phi"]), axis=1)
    sum_j_py = ak.sum(events["post_OR_jet_pt"] * np.sin(events["post_OR_jet_phi"]), axis=1)

    sum_e_px = ak.sum(events["post_OR_elec_pt"] * np.cos(events["post_OR_elec_phi"]), axis=1)
    sum_e_py = ak.sum(events["post_OR_elec_pt"] * np.sin(events["post_OR_elec_phi"]), axis=1)

    met_px = -(sum_j_px + sum_e_px + soft_px)
    met_py = -(sum_j_py + sum_e_py + soft_py)

    events["met_recalc_pt"]  = np.sqrt(met_px**2 + met_py**2)
    events["met_recalc_phi"] = np.arctan2(met_py, met_px)

    tracker.update("MET Rebuilding", events)
    if len(events) == 0: return events

    # ====================================================================
    # 8. dPHI(JET, MET) REQUIREMENT
    # ====================================================================
    dphi = np.abs(events["post_OR_jet_phi"] - events["met_recalc_phi"])
    dphi = ak.where(dphi > np.pi, 2 * np.pi - dphi, dphi)

    pass_dphi = ak.any(dphi < 2.0, axis=1)
    events = events[pass_dphi]

    tracker.update("dPhi(Jet, MET) < 2.0", events)
    if len(events) == 0: return events

    # ====================================================================
    # 9. FINAL COUNTS & TAU VETO
    # ====================================================================
    events["ht"]    = ak.sum(events["post_OR_jet_pt"], axis=1)
    events["n_mu"]  = ak.num(events["post_OR_muon_pt"])
    events["n_ele"] = ak.num(events["post_OR_elec_pt"])
    events["n_tau"] = ak.sum(events["AnalysisTauJetsAuxDyn_JetDeepSetTight"] == 1, axis=1)
    
    events = events[events["n_tau"] == 0]

    tracker.update("Tau Veto / Preselection Complete", events)
    
    return events

# === 3. REGION SPLITTING & SAVING ===
def split_and_save(events, process_name, tracker, batch_suffix, subp_name):
    if len(events) == 0: return

    # Global kinematic masks
    kin_mask_sr = (events["met_recalc_pt"] > 600) & (events["ht"] > 600)
    kin_mask_cr = (events["met_recalc_pt"] > 250) & (events["met_recalc_pt"] <= 600) & (events["ht"] <= 600)
    kin_mask_qcd = (events["met_recalc_pt"] <= 600) & (events["ht"] <= 600)

    # ====================================================================
    # 1. ISOLATED Z-MASS CALCULATION FOR 2L CR
    # ====================================================================
    # First, get the events that pass the base 2L requirements
    base_2L_mask = kin_mask_cr & (events["n_mu"] == 2) & (events["n_bjets"] == 0)
    events_2L = events[base_2L_mask]

    if len(events_2L) > 0:
        # Extract the flat arrays for the leading (0) and subleading (1) muons
        mu1_pt  = events_2L["post_OR_muon_pt"][:, 0]
        mu2_pt  = events_2L["post_OR_muon_pt"][:, 1]
        
        mu1_eta = events_2L["post_OR_muon_eta"][:, 0]
        mu2_eta = events_2L["post_OR_muon_eta"][:, 1]
        
        mu1_phi = events_2L["post_OR_muon_phi"][:, 0]
        mu2_phi = events_2L["post_OR_muon_phi"][:, 1]
        
        deta = mu1_eta - mu2_eta
        dphi = np.abs(mu1_phi - mu2_phi)
        dphi = ak.where(dphi > np.pi, 2 * np.pi - dphi, dphi) # Handle wrap-around
        
        # Calculate invariant mass
        mass_mumu = np.sqrt(2 * mu1_pt * mu2_pt * (np.cosh(deta) - np.cos(dphi)))
        
        # Apply the Z-mass window (66 to 116 GeV)
        events_2L_final = events_2L[(mass_mumu >= 66) & (mass_mumu <= 116)]
    else:
        # Failsafe if the batch had no 2L events
        events_2L_final = events_2L

    # ====================================================================
    # 2. REGION DICTIONARY
    # ====================================================================
    regions = {
        "SR": events[kin_mask_sr & (events["n_mu"] == 0) & (events["n_bjets"] <= 1)],
        
        # 1L CR (W+jets dominated): 1 muon, 0 b-jets
        "CR_1L": events[kin_mask_cr & (events["n_mu"] == 1) & (events["n_bjets"] == 0)],
        
        # 1L1B CR (Top dominated): 1 muon, exactly 1 b-jet
        "CR_1L1B": events[kin_mask_cr & (events["n_mu"] == 1) & (events["n_bjets"] == 1)],

        # 2L CR (Z+jets dominated): 2 muons, 0 b-jets + Invariant Mass Cut
        "CR_2L": events_2L_final,

        # Multijet Reweighting Region (MJRR)
        "CR_MJRR": events[kin_mask_qcd & (events["n_mu"] == 0)]
    }

    # ====================================================================
    # 3. SAVE TO PARQUET
    # ====================================================================
    for reg, data in regions.items():
        tracker.update(f"Region: {reg}", data)
        if len(data) > 0:
            save_path = os.path.join(OUTPUT_DIR, reg)
            os.makedirs(save_path, exist_ok=True)
            file_name = os.path.join(save_path, f"{process_name}_{subp_name}{batch_suffix}.parquet")
            ak.to_parquet(data, file_name)

# === 4. MAIN PIPELINE ===
def process_full_dataset_MC(process_name):
    print(f"--> Processing {process_name}...")
    tracker = CutflowTracker()
    batch_counter = 0 

    subprocesses = os.listdir(os.path.join(BASE_PATH, process_name))
    for subp in subprocesses:

        ttbar_path = os.path.join(BASE_PATH, process_name)
        subp_dir = os.path.join(ttbar_path, f"{subp}")
        print(f"Starting subprocess: {subp}")
        
        if not os.path.exists(subp_dir): continue
        files = sorted([os.path.join(subp_dir, f) for f in os.listdir(subp_dir) if f.endswith(".root")])
        if not files: continue
        
        # Iterate FILE BY FILE so a single crash doesn't kill the subprocess
        for file_path in files:
            try:
                # uproot.iterate now processes just this one file in batches
                for events in uproot.iterate(
                    f"{file_path}:CollectionTree", 
                    BRANCHES, 
                    library="ak", 
                    step_size=BATCH_SIZE
                ):
                    # Clean, Split, and Save as normal
                    cleaned_events = apply_preselection_MC(events, tracker, subp.replace("mc20_13TeV_MC_", ""))
                    split_and_save(cleaned_events, process_name, tracker, f"_batch{batch_counter}", subp.replace("mc20_13TeV_MC_", ""))
                    batch_counter += 1
                    
            except Exception as e:
                # If this specific file is corrupted, log it, and the loop naturally 
                # continues to the next file_path in the 'files' list!
                print(f"  [WARNING] Skipping corrupted file {file_path}")
                print(f"  [ERROR DETAILS] {e}")
                continue 

    tracker.save_csv(process_name, OUTPUT_DIR)
    print(f"<-- Finished {process_name}")

In [11]:
mc_processes = "Diboson", "Multijet", "Single_top", "ttbar", "Wjets", "Zjets"
    
start_time = time.time()
now = datetime.datetime.now()
cpu_cores = multiprocessing.cpu_count()

print("="*60)
print(f"  HEP DATA PROCESSING PIPELINE (BATCHED)")
print(f"  Date:       {now.strftime('%Y-%m-%d')}")
print(f"  System:     {cpu_cores} CPU Cores Detected")
print(f"  Config:     {MAX_WORKERS} Workers | Batch Size: {BATCH_SIZE}")
print("="*60)

# Using ProcessPoolExecutor to bypass the GIL
with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
    executor.map(process_full_dataset_MC, mc_processes)

elapsed = time.time() - start_time
print("="*60)
print(f"  PIPELINE COMPLETE")
print(f"  Total Duration: {elapsed/60:.2f} minutes")
print("="*60)

  HEP DATA PROCESSING PIPELINE (BATCHED)
  Date:       2026-04-09
  System:     128 CPU Cores Detected
  Config:     3 Workers | Batch Size: 500 MB
--> Processing Diboson...--> Processing Single_top...--> Processing Multijet...


Starting subprocess: mc20_13TeV_MC_Sh_2211_WlvZqq
Starting subprocess: mc20_13TeV_MC_Pythia8EvtGen_A14NNPDF23LO_jetjet_JZ9WithSWStarting subprocess: mc20_13TeV_MC_PhPy8EG_A14_tchan_BW50_lept_top

Starting subprocess: mc20_13TeV_MC_Pythia8EvtGen_A14NNPDF23LO_jetjet_JZ4WithSW
Starting subprocess: mc20_13TeV_MC_Sh_2211_WlvWqq
Starting subprocess: mc20_13TeV_MC_PowhegPythia8EvtGen_A14_singletop_schan_lept_antitop
Starting subprocess: mc20_13TeV_MC_PowhegPythia8EvtGen_A14_singletop_schan_lept_top
Starting subprocess: mc20_13TeV_MC_PhPy8EG_A14_tchan_BW50_lept_antitop
Saved cutflow for Single_top
<-- Finished Single_top
--> Processing ttbar...
Starting subprocess: mc20_13TeV_MC_PhPy8EG_A14_ttbar_hdamp258p75_allhad
Starting subprocess: mc20_13TeV_MC_Sh_2211_WqqZvv
Sta